# Notebook 05 – Measuring Polarisation

**DigitAfrica Workshop 2026 – Identifying Epistemic Enclaves and Understanding Polarisation**

## Learning objectives
- Compute structural polarisation metrics (e.g. Random Walk Controversy, Boundary Edges)
- Compute content-based polarisation via opinion scores
- Compare polarisation across different network slices

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Reconstruct the partitioned network

In [ ]:
rng = np.random.default_rng(1)
n_nodes = 100
groups = {i: ("A" if i < 50 else "B") for i in range(n_nodes)}

G = nx.Graph()
G.add_nodes_from(groups.keys())
nx.set_node_attributes(G, groups, "group")

for u in range(n_nodes):
    for v in range(u + 1, n_nodes):
        same_group = groups[u] == groups[v]
        prob = 0.08 if same_group else 0.01
        if rng.random() < prob:
            G.add_edge(u, v)

## 2. Boundary edge ratio

A simple structural measure: the fraction of edges that cross community boundaries.

In [ ]:
total_edges = G.number_of_edges()
cross_edges = sum(
    1 for u, v in G.edges()
    if groups[u] != groups[v]
)
boundary_ratio = cross_edges / total_edges
print(f"Cross-community edges : {cross_edges} / {total_edges}")
print(f"Boundary edge ratio   : {boundary_ratio:.4f}")
print(f"(lower → more polarised)")

## 3. Opinion-based polarisation

In [ ]:
# Assign synthetic opinion scores
opinion = {
    n: rng.normal(-0.6 if groups[n] == "A" else 0.6, 0.15)
    for n in G.nodes()
}
nx.set_node_attributes(G, opinion, "opinion")

scores_a = [opinion[n] for n in G.nodes() if groups[n] == "A"]
scores_b = [opinion[n] for n in G.nodes() if groups[n] == "B"]

print(f"Mean opinion – Group A: {np.mean(scores_a):.3f}")
print(f"Mean opinion – Group B: {np.mean(scores_b):.3f}")
print(f"Opinion gap: {abs(np.mean(scores_b) - np.mean(scores_a)):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores_a, bins=20, alpha=0.6, label="Group A", color="steelblue")
ax.hist(scores_b, bins=20, alpha=0.6, label="Group B", color="tomato")
ax.axvline(np.mean(scores_a), color="steelblue", linestyle="--")
ax.axvline(np.mean(scores_b), color="tomato", linestyle="--")
ax.set_xlabel("Opinion score")
ax.set_ylabel("Count")
ax.set_title("Opinion distribution by group")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Exercise

- Increase the cross-group edge probability and observe how the boundary ratio changes.
- How does the opinion gap change when you introduce more bridging edges?